# 🚀 Retrain LoRA SigLIP2-SO400M on `folds_v3.csv`
Notebook ini dirancang untuk menjalankan retraining LoRA menggunakan dataset hasil pembersihan tahap 3 (`folds_v3.csv`). Dataset akan dicopy secara otomatis ke penyimpanan lokal Colab SSD agar I/O rate cepat (32 workers) tanpa bottleneck Google Drive.

In [2]:
import os, sys, shutil, glob
from concurrent.futures import ThreadPoolExecutor
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# 2. Clone atau Pull Repo Terbaru
REPO_DIR = '/content/satria-data-bdcugm02'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/agaggigit/satria-data-bdcugm02.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

# 3. Upgrade Torchao untuk PEFT & Install Dependensi Pinned
!pip install -q -U "torchao>=0.16.0"
!pip install -q --no-warn-conflicts -r {REPO_DIR}/track_b/requirements.txt

# 4. Copy Cepat Gambar Dataset ke Storage Lokal Colab (/tmp) dengan 32 Workers
DRIVE_TRAIN_DIR = '/content/drive/MyDrive/BDC2026/train'
LOCAL_TRAIN_DIR = '/tmp/dataset/train'

def copy_img_worker(args):
    src, dst = args
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if not os.path.exists(dst) or os.path.getsize(src) != os.path.getsize(dst):
        shutil.copy2(src, dst)

if os.path.exists(DRIVE_TRAIN_DIR):
    print("🚀 Memulai copy cepat SELURUH GAMBAR ke storage lokal Colab (/tmp) dengan 32 workers...")
    all_imgs = []
    for root, _, files in os.walk(DRIVE_TRAIN_DIR):
        for f in files:
            if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp')):
                all_imgs.append(os.path.join(root, f))

    files_to_copy = [(src, os.path.join(LOCAL_TRAIN_DIR, os.path.relpath(src, DRIVE_TRAIN_DIR))) for src in all_imgs]

    with ThreadPoolExecutor(max_workers=32) as executor:
        list(executor.map(copy_img_worker, files_to_copy))
    print(f"✅ Selesai meng-copy {len(files_to_copy)} gambar ke storage lokal Colab: {LOCAL_TRAIN_DIR}")
else:
    print("⚠️ Folder gambar Drive belum ditemukan di path standar.")


Mounted at /content/drive
Cloning into '/content/satria-data-bdcugm02'...
remote: Enumerating objects: 883, done.
remote: Counting objects: 100% (200/200), done.
remote: Compressing objects: 100% (150/150), done.
remote: Total 883 (delta 114), reused 106 (delta 50), pack-reused 683 (from 1)
Receiving objects: 100% (883/883), 41.38 MiB | 50.86 MiB/s, done.
Resolving deltas: 100% (475/475), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 48.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.5/47.5 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 71.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.5/

## Training LoRA dengan Data `v3`
Jalankan cell di bawah ini untuk memulai proses *smoke test* atau *training*.

In [1]:
import sys, importlib
sys.path.insert(0, '/content/satria-data-bdcugm02/track_a/src')
sys.path.insert(0, '/content/satria-data-bdcugm02/track_b/src')
sys.path.insert(0, '/content/satria-data-bdcugm02/track_b/experiments')

import embed, lora_ft, config
importlib.reload(config)
importlib.reload(embed)
importlib.reload(lora_ft)

from config import CFG, make_cfg

VARIANT = 'lora'
CHECKPOINT = 'google/siglip2-so400m-patch14-384'

# Gunakan folds_v3_csv untuk run ini
cfg = make_cfg(
    run_name=f'{VARIANT}_ft_fold0_5ep_v3',
    folds_csv=CFG.folds_v3_csv,   # <-- Train & evaluasi hanya di v3
    batch=8,
    accum_steps=4
)

# Menjalankan 5 Epoch LoRA (Single Validation)
result = lora_ft.run_smoke_test_fold0(
    variant=VARIANT,
    cfg=cfg,
    checkpoint=CHECKPOINT,
    max_epochs=5,
    n_last_blocks=4
)
result


config.json:   0%|          | 0.00/559 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/394 [00:00<?, ?B/s]

Aligning cfg.img_size to checkpoint native size: 384
⚡ [Fast I/O] Path gambar dialihkan ke lokal SSD Colab: /tmp/dataset/train
[WasteDataset] Loaded 20666 sampel | dist: {0: 7781, 1: 3154, 2: 9731}
[WasteDataset] Loaded 5160 sampel | dist: {0: 1957, 1: 779, 2: 2424}

--- Verifikasi trainable parameters [lora] ---
trainable params: 73,728 || all params: 428,299,328 || trainable%: 0.0172
  [lora] epoch 1/5 | train_loss 0.2243 | val_f1 0.9859 | 25.7 mnt
  [lora] epoch 2/5 | train_loss 0.0332 | val_f1 0.9933 | 25.9 mnt
  [lora] epoch 3/5 | train_loss 0.0218 | val_f1 0.9925 | 25.9 mnt
  [lora] epoch 4/5 | train_loss 0.0213 | val_f1 0.9946 | 25.9 mnt
  [lora] epoch 5/5 | train_loss 0.0165 | val_f1 0.9953 | 25.9 mnt

[lora] minutes_per_epoch = 25.87
[lora] BEST Epoch: 5 | Best val_f1: 0.9953
[lora] Estimasi 5-fold x 5 epoch = 10.78 jam GPU


{'variant': 'lora',
 'best_epoch': 5,
 'best_val_f1': 0.9953159250578012,
 'last_val_f1': 0.9953159250578012,
 'minutes_per_epoch': 25.872448185284934,
 'est_5fold_hours': 10.780186743868722}